## **Fresh Eyes on CMIP Data Workshop 2026**: Application with Julia

#### **Setup**
1. Download Julia from https://github.com/JuliaLang/juliaup#installation
2. Install Julia extension in Visual Code Studio market place
3. Code can be execute in 
    - Julia Read-Eval-Print-Loop (Julia: Start REPL)
    - Jupyter Notebook
    - ...
4. When selecting the Kernel in Jupyter:
    - Julia release channel (automatically uses most recent version of Julia on your machine)


In [1]:
print("Hello World")

Hello World

#### **Package management**
- For different projects you often need different additional packages or libraries
- In Julia the `pkg` helps you to manage libraries and make your project reproducible for others
    - You have a global environment in `~/.julia/environments/v1.13/` (in case of version 1.13)
    - You should create a local environment in your project folder
    - All packages are installed into the same directory `~/.julia/packages/`, independent of the environment
- The environent contains two files
    - `Project.toml`
    - `Manifest.toml`

In [ ]:
using Pkg # tell Julia to use pkg manager
Pkg.activate() # activate the global environment (should already be the case)
Base.active_project() # show which Project.toml file is currently active

  Activating project at `~/.julia/environments/v1.13`


"/Users/tim/.julia/environments/v1.13/Project.toml"

In [3]:
print(Base.load_path()) # shows all paths to environments that are currently checked when loading packages
print("\n") # print a linebreak
print(LOAD_PATH) # print LOAD_PATH variable, showing all environments which will currently be checked

["/Users/tim/.julia/environments/v1.13/Project.toml", "/Users/tim/.julia/juliaup/julia-1.13.1+0.aarch64.apple.darwin14/Julia-1.13.app/Contents/Resources/julia/share/julia/stdlib/v1.13"]
["@", "@v#.#", "@stdlib"]

In [4]:
Pkg.add("BenchmarkTools") # install BenchmarkTools package to global environment

     Project No packages added to or removed from `~/.julia/environments/v1.13/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.13/Manifest.toml`


In [5]:
Pkg.activate(".") # activate current directory as project environment
using BenchmarkTools # install BenchmarkTools (global environment will be used)

  Activating project at `~/Documents/GitHub/Julia-CMIP`


In [18]:
deleteat!(LOAD_PATH, 2) # remove the global environment from LOAD_PATH so that BenchmarkTools is not found when using it in this notebook
using BenchmarkTools # this should now throw an error since BenchmarkTools is not installed in the current environment

ArgumentError: ArgumentError: Package BenchmarkTools not found in current path, maybe you meant `import/using .BenchmarkTools`.
- Otherwise, run `import Pkg; Pkg.add("BenchmarkTools")` to install the BenchmarkTools package.

In [6]:
Pkg.add("BenchmarkTools") # install BenchmarkTools package to local environment (create Project.toml and Manifest.toml files)
using BenchmarkTools

     Project No packages added to or removed from `~/Documents/GitHub/Julia-CMIP/Project.toml`
    Manifest No packages added to or removed from `~/Documents/GitHub/Julia-CMIP/Manifest.toml`


In [7]:
# Aftere cloning a repository, you run Pkg.instantiate() to install all dependencies listed in the Project.toml 
# and Manifest.toml files. This will ensure that your local environment has all the necessary packages to run the 
# code in the repository.
Pkg.instantiate()

#### **Why (and when) Julia?**

- Both python and julia are high level languages (human readible) while processors can only read machine code
- Both languages use dynamics typing, so variables types do not need to be defiend beforehand
- However, they differ in the way how code is translated to machine code
    - **Python** is an interpreter languge compiles code into intermediate bytecode (saved in `__pycache__`), which the interpreter then executes step-by-step at runtime without compiling it into native CPU machine code.
    - **Julia** uses JIT (just in time) compilation, where a function is compiled just before it is called (possible through multiple dispatch)
    - In **C** the whole program code is compiled before execution
- Python has functions that are precompiled in **C**
    - If you use only these functions, python will not be slower
    - If you use a lot of custom functions with loops and conditionals, they will be faster in Julia
- Generally
    - Use python for deep learning, tabular data processing (pandas) or general software engineering (object oriented)
    - Use julia for scientific coding (differential equations, ABMs, physical models, ...) and mathematics and optimisation

#### Example
Calculate the eucledian distance on two vectors with 10 Mio elements ($d=\sqrt{\sum(a_i-b_i)^2}$).
1. Using native python
2. Using numpy (precompiled functions in C)
3. Using native julia

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using BenchmarkTools
using PythonCall  # Package that allows to call Python from Julia (loads numpy library as specified in CondaPkg.toml)

np = pyimport("numpy") # import numpy library from Python

  Activating project at `~/Documents/GitHub/Julia-CMIP`


Python: <module 'numpy' from '/Users/tim/Documents/GitHub/Julia-CMIP/.CondaPkg/.pixi/envs/default/lib/python3.12/site-packages/numpy/__init__.py'>

In [ ]:
# Native Python Function (Runs in Python's bytecode interpreter)
@pyexec """
def euclidean_native_py(x, y):
    total = 0.0
    for i in range(len(x)):
        diff = x[i] - y[i]
        total += diff * diff
    return total ** 0.5
""" => euclidean_native_py

# Native Julia Function (Plain loop, JIT-compiled to native CPU machine code)
function euclidean_native_jl(x, y)
    total = 0.0
    for i in eachindex(x, y)
        diff = x[i] - y[i]
        total += diff * diff
    end
    return sqrt(total)
end

# Native Julia Function (Plain loop, JIT-compiled to native CPU machine code)
function euclidean_native_jl_fast(x, y)
    total = 0.0
    # @ is a macro that optimizes the compilation of the loop
    # @inbound tells the compiler to skip bounds checking for array indexing, which can improve performance
    # @simd, Single Instruction, Multiple Data, lets the compiler parallelize the operation
    @inbounds @simd for i in eachindex(x, y)
        diff = x[i] - y[i]
        total += diff * diff
    end
    return sqrt(total)
end

euclidean_native_jl_fast (generic function with 1 method)

In [ ]:
# create random array of length N with Float64 values between 0 and 1
N = 10_000_000
a_jl = rand(Float64, N)
b_jl = rand(Float64, N)

# Convert data into Python list and NumPy array formats
a_pylist = pylist(a_jl);       b_pylist = pylist(b_jl);
a_numpy  = np.array(a_jl);     b_numpy  = np.array(b_jl);

# @btime is a macro from BenchmarkTools.jl that measures the execution time of a function call
println("1. Native Python (Interpreted for-loop):")
@btime $euclidean_native_py($a_pylist, $b_pylist)

println("\n2. Python + NumPy (Pre-compiled C library):")
@btime $np.linalg.norm($a_numpy - $b_numpy)

println("\n3. Native Julia (Pure Julia for-loop, JIT compiled):")
@btime euclidean_native_jl($a_jl, $b_jl)

println("\n4. Native Julia (Optimized for-loop, JIT compiled):")
@btime euclidean_native_jl_fast($a_jl, $b_jl);

1. Native Python (Interpreted for-loop):
  377.426 ms (2 allocations: 32 bytes)

2. Python + NumPy (Pre-compiled C library):
  11.788 ms (9 allocations: 176 bytes)

3. Native Julia (Pure Julia for-loop, JIT compiled):
  9.373 ms (0 allocations: 0 bytes)

3. Native Julia (Pure Julia for-loop, JIT compiled):
  9.373 ms (0 allocations: 0 bytes)

4. Native Julia (Optimized for-loop, JIT compiled):
  2.595 ms (0 allocations: 0 bytes)


In [ ]:
# Capture the outputs
# Convert Python objects (Py) to native Julia Float64 using pyconvert
res_py    = pyconvert(Float64, euclidean_native_py(a_pylist, b_pylist))
res_numpy = pyconvert(Float64, np.linalg.norm(a_numpy - b_numpy))
res_jl    = euclidean_native_jl(a_jl, b_jl)
res_jl_fast = euclidean_native_jl_fast(a_jl, b_jl)

# Print the exact values and their differences
println("Native Python: ", res_py)
println("Python NumPy:  ", res_numpy)
println("Native Julia:  ", res_jl)
println("Julia (Optmz): ", res_jl_fast)

# Check approximate equality (using isapprox or the ≈ symbol)
println("\nAre all four approximately equal? ", res_py ≈ res_numpy ≈ res_jl ≈ res_jl_fast)
println("Are Python and Julia strictly identical (==)? ", res_py == res_jl)

Native Python: 1290.8442164820844
Python NumPy:  1290.8442164820888
Native Julia:  1290.8442164820844
Julia (Optmz): 1290.8442164820879

Are all four approximately equal? true
Are Python and Julia strictly identical (==)? true


**Things to Notice**
1. **Native Python is slowest**: As expected, the pure Python loop takes the longest because the CPython interpreter has to check variable types and create new float objects in memory on every single iteration.
2. **Native Julia is even faster than NumPy**: even though NumPy also used pre-compiled C code, it is slower than Julia, becuase it evaluates the function `np.linalg.norm(a-b)` in two steps. First it runs the $a-b$ statement and writes it temporarily into RAM (9 allocations), then it runs the NumPy function. This extra step is what makes it slower than Julia which executes everything as a single pass without allocating temporary arrays. 
3. **@inbound and @simd**: both optimization speed up the Julia implementation 5 times compared to the basic Julia implementation without macros.
4. **Warm-Up**: The very first time you run a Julia function, there is a brief pause while Julia's JIT compiler translates the code into native machine instructions. The @btime macro excludes this one-time compilation overhead by executing an untimed warm-up run first and then reporting the minimum execution time across multiple runs.
5. **Precision**: A basic sequential loop in Python and a basic sequential loop in Julia produce bit-for-bit identical results (==) because they add the numbers in the exact same left-to-right order. In contrast, NumPy and the @simd-optimized Julia loop use SIMD vectorization (accumulating multiple partial sums in parallel inside CPU registers). Changing the order of addition slightly alters the 13th–15th decimal places, and accumulating into separate partial sums actually reduces floating-point rounding error.
